# **CA-STM - Combined Action - Softened Truss Model**

Algoritmo para o cálculo do comportamento carga-deformação em seções de concreto armado, usando o CA-STM (Combined Action - Softened Truss Model) proposto por GREENE (2006) e o procedimento de solução do problema não-linear de SILVA (2016).

**REFERÊNCIAS**

*   GREENE, G. G. Jr., 2006. “Behavior of reinforced concrete girders under cyclic torsion and torsion combined with shear: experimental investigation and analytical models”, Ph.D.Dissertation, Department of Civil Engineering, University of MissouriRolla, USA.
*   SILVA, J. R. B., 2016. “Procedimento Eficiente para Análise de Seções em Concreto Armado usando o Modelo de Treliça com Amolecimento”, Dissertação de Mestrado, Programa de pós-graduação em Engenharia Civil, Universidade Federal de Pernambuco, Brasil.
*   MCMULLEN, A. E.; WARWARUK, J. M, 1970. “Concrete Beams in Bending, Torsion and Shear”, Journal of Structural Engineering, ASCE, V. 96, No. ST5, pp. 885-903.
*   LAMPERT, P.; THÜRLIMANN, B., 1968. “Torsion Tests of Reinforced Concrete Beams,” Report No. 6506-2, Intitut für Baustatik, ETH, Zurich, 101 pp.
*   RAHAL, K. N.; COLLINS, M. P., 1995. “Effect of the Thickness of Concrete Cover on the Shear-Torsion Interaction – an Experimental Investigation,” ACI Structural Journal, V. 92, No. 3, pp. 334-342.

In [ ]:
### Importações:
import numpy as np
import pandas as pd
from tabulate import tabulate
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from scipy.optimize import least_squares

### **1. Dados de entrada**

#### Experimento 2-1 de McMullen e Warwaruk (1970)

In [ ]:
### Geometria da seção:
h = 0.30                             # Altura da seção (m)
b = 0.15                             # Base da seção (m)
c1 = 0.04                            # Distância entre o eixo da barra longitudinal e a face lateral (m)
t1, t2, t3, t4 = b/2, b/2, b/2, b/2  # Espessura máxima dos painéis (m)

### Quantidade das armaduras:
Al1 = 2.8353*10**-4  # Área da armadura longitudinal no painel 1 (m2)
Al2 = 2.8353*10**-4  # Área da armadura longitudinal no painel 2 (m2)
Al3 = 2.8353*10**-4  # Área da armadura longitudinal no painel 3 (m2)
Al4 = 2.8353*10**-4  # Área da armadura longitudinal no painel 4 (m2)
At  = 7.8540*10**-5  # Área da armadura transversal (m2)
s = 0.083            # Espaçamento da armadura transversal (m)

### Propriedades Mecânicas dos aços:
fyk = 500    # Tensão de escoamento do aço de projeto (MPa)
fly = 344    # Tensão de escoamento da armadura longitudinal (MPa)
fty = 344    # Tensão de escoamento da armadura transversal (MPa)
Es = 200000  # Módulo de elasticidade dos aços (MPa)
gammas = 1.15

### Propriedades Mecânicas do concreto:
fck = 34        # Resistência característica do concreto comprimido (MPa)
e0 = -2*10**-3  # Deformação de compressão última do concreto (1/1000)
ecr = 0.1       # Deformação de tração que o concreto fissura (1/1000)
ecr0 = 4.5      # Deformação de tração limite do concreto (1/1000)
eds1 = -0.01    # Deformação de compressão inicial do painel 1 (1/1000)
gammac = 1.4

### Relação dos outros esforços com o momento torsor:
MyTx = 0  # Momento Fletor y / Momento Torsor x
MzTx = 0  # Momento Fletor z / Momento Torsor x
VyTx = 0  # Esforço Cortante y / Momento Torsor x
VzTx = 0  # Esforço Cortante z / Momento Torsor x
NxTx = 0  # Esforço Normal x / Momento Torsor x

### Resultados experimentais:
thetaexp = np.array([0.000000, 0.002158, 0.009831, 0.017744,
                     0.025897, 0.035009, 0.04484, 0.058268])
Texp = np.array([ 0.0000,  4.6999,  6.8146,  9.1158,
                 11.3548, 13.6561, 15.8329, 18.1963])

### Curva da dissertação:
thetapub = np.array([0.0000, 0.0033, 0.0051, 0.0073, 0.0120, 0.0157, 0.0189,
                     0.0230, 0.0270, 0.0316, 0.0362, 0.0414, 0.0461, 0.0529,
                     0.0597, 0.0630, 0.0654])
Tpub = np.array([ 0.0000,  7.6923,  7.9320,  8.1318,  9.2707, 10.1698, 10.8691,
                 11.6083, 12.2477, 12.9470, 13.6063, 14.2457, 14.8251, 15.4845,
                 15.9040, 16.0639, 16.0639])

### Resultado adicionais simulados no MATLAB em 24/05/2023 (TolX = TolFun =10^-22):
thetaNewM = np.array([0.0000, 0.0003, 0.0005, 0.0006, 0.0008, 0.0010, 0.0011,
                      0.0013, 0.0014, 0.0016, 0.0017, 0.0019, 0.0021, 0.0022,
                      0.0024, 0.0025, 0.0026, 0.0028, 0.0029, 0.0030, 0.0035,
                      0.0040, 0.0044, 0.0048, 0.0052, 0.0056, 0.0059, 0.0063,
                      0.0066, 0.0070, 0.0073, 0.0076, 0.0080, 0.0083, 0.0086,
                      0.0090, 0.0093, 0.0096, 0.0099, 0.0102, 0.0105, 0.0109,
                      0.0112, 0.0115, 0.0118, 0.0121, 0.0124, 0.0127, 0.0130,
                      0.0133, 0.0136, 0.0139, 0.0142, 0.0145, 0.0148, 0.0151,
                      0.0154, 0.0156, 0.0159, 0.0162, 0.0165, 0.0168, 0.0171,
                      0.0174, 0.0176, 0.0179, 0.0182, 0.0185, 0.0188, 0.0191,
                      0.0193, 0.0196, 0.0199, 0.0202, 0.0204, 0.0207, 0.0210,
                      0.0213, 0.0215, 0.0218, 0.0221, 0.0223, 0.0226, 0.0229,
                      0.0231, 0.0234, 0.0237, 0.0239, 0.0242, 0.0245, 0.0247,
                      0.0250, 0.0253, 0.0255, 0.0258, 0.0260, 0.0263, 0.0266,
                      0.0268, 0.0271, 0.0273, 0.0276, 0.0279, 0.0281, 0.0284,
                      0.0286, 0.0289, 0.0291, 0.0294, 0.0296, 0.0299, 0.0301,
                      0.0304, 0.0306, 0.0309, 0.0311, 0.0314, 0.0316, 0.0319,
                      0.0321, 0.0324, 0.0326, 0.0329, 0.0331, 0.0334, 0.0336,
                      0.0338, 0.0341, 0.0343, 0.0346, 0.0348, 0.0351, 0.0353,
                      0.0355, 0.0358, 0.0360, 0.0363, 0.0365, 0.0367, 0.0370,
                      0.0372, 0.0374, 0.0377, 0.0379, 0.0382, 0.0384, 0.0386,
                      0.0389, 0.0391, 0.0393, 0.0396, 0.0398, 0.0400, 0.0403,
                      0.0405, 0.0407, 0.0409, 0.0412, 0.0414, 0.0416, 0.0419,
                      0.0421, 0.0423, 0.0425, 0.0428, 0.0430, 0.0432, 0.0434,
                      0.0437, 0.0439, 0.0441, 0.0443, 0.0446, 0.0448, 0.0450,
                      0.0452, 0.0455, 0.0457, 0.0459, 0.0461, 0.0463, 0.0466,
                      0.0468, 0.0470, 0.0472, 0.0474, 0.0476, 0.0479, 0.0481,
                      0.0483, 0.0485, 0.0487, 0.0489, 0.0492, 0.0494, 0.0496,
                      0.0498, 0.0500, 0.0502, 0.0504, 0.0507, 0.0509, 0.0511,
                      0.0513, 0.0515, 0.0517, 0.0519, 0.0521, 0.0523, 0.0525,
                      0.0527, 0.0530, 0.0532, 0.0534, 0.0536, 0.0538, 0.0540,
                      0.0542, 0.0544, 0.0546, 0.0548, 0.0550, 0.0552, 0.0554,
                      0.0556, 0.0558, 0.0560, 0.0562, 0.0564, 0.0566, 0.0568,
                      0.0570, 0.0572, 0.0574, 0.0576, 0.0578, 0.0580, 0.0582,
                      0.0584, 0.0586, 0.0588, 0.0590, 0.0592, 0.0594, 0.0596,
                      0.0598, 0.0599, 0.0601, 0.0603, 0.0605, 0.0607, 0.0609,
                      0.0611, 0.0613, 0.0615, 0.0617, 0.0619, 0.0620, 0.0622,
                      0.0624, 0.0626, 0.0628, 0.0630, 0.0632, 0.0633, 0.0635,
                      0.0637, 0.0639, 0.0641, 0.0643, 0.0644, 0.0646, 0.0648,
                      0.0650, 0.0652, 0.0653, 0.0655, 0.0657, 0.0659, 0.0661,
                      0.0662, 0.0664])

TNewM = np.array([ 0.0000,  0.7878,  1.1781,  1.5661,  1.9519,  2.3353,  2.7165,
                   3.0954,  3.4721,  3.8466,  4.2189,  4.5890,  4.9570,  5.3228,
                   5.6865,  6.0216,  6.3320,  6.6295,  6.9150,  7.1890,  7.6072,
                   7.6215,  7.6251,  7.6308,  7.6423,  7.6603,  7.6846,  7.7144,
                   7.7490,  7.7879,  7.8304,  7.8760,  7.9243,  7.9749,  8.0274,
                   8.0816,  8.1373,  8.1942,  8.2521,  8.3109,  8.3705,  8.4307,
                   8.4915,  8.5527,  8.6142,  8.6761,  8.7381,  8.8004,  8.8628,
                   8.9252,  8.9877,  9.0502,  9.1126,  9.1750,  9.2373,  9.2995,
                   9.3616,  9.4235,  9.4853,  9.5469,  9.6083,  9.6694,  9.7304,
                   9.7911,  9.8516,  9.9119,  9.9719, 10.0316, 10.0911, 10.1503,
                  10.2092, 10.2679, 10.3262, 10.3843, 10.4420, 10.4995, 10.5566,
                  10.6135, 10.6700, 10.7263, 10.7822, 10.8378, 10.8931, 10.9481,
                  11.0028, 11.0571, 11.1111, 11.1648, 11.2182, 11.2713, 11.3241,
                  11.3765, 11.4286, 11.4804, 11.5318, 11.5830, 11.6338, 11.6843,
                  11.7344, 11.7843, 11.8338, 11.8830, 11.9319, 11.9804, 12.0287,
                  12.0766, 12.1242, 12.1714, 12.2184, 12.2650, 12.3113, 12.3573,
                  12.4030, 12.4484, 12.4934, 12.5381, 12.5825, 12.6266, 12.6704,
                  12.7139, 12.7570, 12.7999, 12.8424, 12.8846, 12.9265, 12.9681,
                  13.0093, 13.0503, 13.0910, 13.1313, 13.1714, 13.2111, 13.2505,
                  13.2896, 13.3285, 13.3670, 13.4052, 13.4431, 13.4807, 13.5180,
                  13.5550, 13.5917, 13.6281, 13.6642, 13.7000, 13.7355, 13.7707,
                  13.8056, 13.8402, 13.8745, 13.9085, 13.9422, 13.9757, 14.0088,
                  14.0416, 14.0742, 14.1065, 14.1384, 14.1701, 14.2015, 14.2326,
                  14.2634, 14.2940, 14.3242, 14.3542, 14.3838, 14.4132, 14.4423,
                  14.4711, 14.4997, 14.5279, 14.5559, 14.5836, 14.6110, 14.6381,
                  14.6650, 14.6916, 14.7179, 14.7439, 14.7696, 14.7951, 14.8203,
                  14.8452, 14.8699, 14.8942, 14.9183, 14.9421, 14.9657, 14.9890,
                  15.0120, 15.0347, 15.0572, 15.0794, 15.1013, 15.1230, 15.1444,
                  15.1655, 15.1864, 15.2070, 15.2273, 15.2474, 15.2672, 15.2867,
                  15.3060, 15.3250, 15.3437, 15.3622, 15.3805, 15.3984, 15.4161,
                  15.4336, 15.4508, 15.4677, 15.4844, 15.5008, 15.5170, 15.5329,
                  15.5485, 15.5639, 15.5790, 15.5939, 15.6085, 15.6229, 15.6370,
                  15.6509, 15.6645, 15.6779, 15.6910, 15.7039, 15.7165, 15.7288,
                  15.7410, 15.7528, 15.7644, 15.7758, 15.7869, 15.7978, 15.8084,
                  15.8188, 15.8289, 15.8388, 15.8484, 15.8578, 15.8670, 15.8759,
                  15.8845, 15.8929, 15.9011, 15.9090, 15.9167, 15.9242, 15.9314,
                  15.9383, 15.9450, 15.9515, 15.9577, 15.9637, 15.9695, 15.9750,
                  15.9803, 15.9853, 15.9901, 15.9946, 15.9989, 16.0030, 16.0069,
                  16.0105, 16.0138, 16.0169, 16.0198, 16.0225, 16.0249, 16.0271,
                  16.0290, 16.0307, 16.0321, 16.0334, 16.0344, 16.0351, 16.0356,
                  16.0359, 16.0360])

### Rigidezes da dissertação (kN.m2):
RigTeoEPub = 2680
RigExpEPub = 2177
RigNumEPub = 2293
RigExpUPub = 312
RigNumUPub = 243

### Divisores da dissertação:
DivExpPub = 7.0
DivNumPub = 9.4

### Rótulos para plotagem da curva:
LabelNum  = 'Código Google Colab'
LabelPub  = 'Dissertação de mestrado - Silva (2016)'
LabelNewM = 'Nova análise via MATLAB'
LabelExp  = 'Experimento 2-1 de McMullen e Warwaruk (1970)'

#### CLASSE DE AGRESSIVIDADE AMBIENTAL I - CAAI

In [ ]:
### Geometria da seção:
h = 0.2                             # Altura da seção (m)
b = 0.12                             # Base da seção (m)
cob = 0.025
c1 = cob + 0.005                     # Distância entre o eixo da barra longitudinal e a face lateral (m)
t1, t2, t3, t4 = b/2, b/2, b/2, b/2  # Espessura máxima dos painéis (m)

### Quantidade das armaduras:
s = 1            # Espaçamento da armadura transversal (m)

### Propriedades Mecânicas dos aços:
fyk = 500    # Tensão de escoamento do aço de projeto (MPa)
fly = 500    # Tensão de escoamento da armadura longitudinal (MPa)
fty = 500    # Tensão de escoamento da armadura transversal (MPa)
Es = 210000  # Módulo de elasticidade dos aços (MPa)
gammas = 1.15

### Propriedades Mecânicas do concreto:
fck = 20        # Resistência característica do concreto comprimido (MPa)
e0 = -2*10**-3  # Deformação de compressão última do concreto (1/1000)
ecr = 0.1       # Deformação de tração que o concreto fissura (1/1000)
ecr0 = 4.5      # Deformação de tração limite do concreto (1/1000)
eds1 = -0.01    # Deformação de compressão inicial do painel 1 (1/1000)
gammac = 1.4

### Relação dos outros esforços com o momento torsor:
MyTx = 0  # Momento Fletor y / Momento Torsor x
MzTx = 0  # Momento Fletor z / Momento Torsor x
#VyTx = 0  # Esforço Cortante y / Momento Torsor x
VzTx = 0  # Esforço Cortante z / Momento Torsor x
NxTx = 0  # Esforço Normal x / Momento Torsor x

#### CLASSE DE AGRESSIVIDADE AMBIENTAL II - CAAII

In [ ]:
### Geometria da seção:
h = 0.2                              # Altura da seção (m)
b = 0.12                             # Base da seção (m)
cob = 0.03
c1 = cob + 0.005                     # Distância entre o eixo da barra longitudinal e a face lateral (m)
t1, t2, t3, t4 = b/2, b/2, b/2, b/2  # Espessura máxima dos painéis (m)

### Quantidade das armaduras:
s = 1            # Espaçamento da armadura transversal (m)

### Propriedades Mecânicas dos aços:
fyk = 500    # Tensão de escoamento do aço de projeto (MPa)
fly = 500    # Tensão de escoamento da armadura longitudinal (MPa)
fty = 500    # Tensão de escoamento da armadura transversal (MPa)
Es = 210000  # Módulo de elasticidade dos aços (MPa)
gammas = 1.15

### Propriedades Mecânicas do concreto:
fck = 25        # Resistência característica do concreto comprimido (MPa)
e0 = -2*10**-3  # Deformação de compressão última do concreto (1/1000)
ecr = 0.1       # Deformação de tração que o concreto fissura (1/1000)
ecr0 = 4.5      # Deformação de tração limite do concreto (1/1000)
eds1 = -0.01    # Deformação de compressão inicial do painel 1 (1/1000)
gammac = 1.4

### Relação dos outros esforços com o momento torsor:
MyTx = 0  # Momento Fletor y / Momento Torsor x
MzTx = 0  # Momento Fletor z / Momento Torsor x
#VyTx = 0  # Esforço Cortante y / Momento Torsor x
VzTx = 0  # Esforço Cortante z / Momento Torsor x
NxTx = 0  # Esforço Normal x / Momento Torsor x

#### CLASSE DE AGRESSIVIDADE AMBIENTAL III - CAAIII

In [ ]:
### Geometria da seção:
h = 0.20                             # Altura da seção (m)
b = 0.12                             # Base da seção (m)
cob = 0.04
c1 = cob + 0.005                     # Distância entre o eixo da barra longitudinal e a face lateral (m)
t1, t2, t3, t4 = b/2, b/2, b/2, b/2  # Espessura máxima dos painéis (m)

### Quantidade das armaduras:
s = 1            # Espaçamento da armadura transversal (m)

### Propriedades Mecânicas dos aços:
fyk = 500    # Tensão de escoamento do aço de projeto (MPa)
fly = 500    # Tensão de escoamento da armadura longitudinal (MPa)
fty = 500    # Tensão de escoamento da armadura transversal (MPa)
Es = 210000  # Módulo de elasticidade dos aços (MPa)
gammas = 1.15

### Propriedades Mecânicas do concreto:
fck = 30        # Resistência característica do concreto comprimido (MPa)
e0 = -2*10**-3  # Deformação de compressão última do concreto (1/1000)
ecr = 0.1       # Deformação de tração que o concreto fissura (1/1000)
ecr0 = 4.5      # Deformação de tração limite do concreto (1/1000)
eds1 = -0.01    # Deformação de compressão inicial do painel 1 (1/1000)
gammac = 1.4

### Relação dos outros esforços com o momento torsor:
MyTx = 0  # Momento Fletor y / Momento Torsor x
MzTx = 0  # Momento Fletor z / Momento Torsor x
#VyTx = 0  # Esforço Cortante y / Momento Torsor x
VzTx = 0  # Esforço Cortante z / Momento Torsor x
NxTx = 0  # Esforço Normal x / Momento Torsor x

#### CLASSE DE AGRESSIVIDADE AMBIENTAL IV - CAAIV

In [ ]:
### Geometria da seção:
h = 0.40                             # Altura da seção (m)
b = 0.2                             # Base da seção (m)
cob = 0.05
c1 = cob + 0.005                     # Distância entre o eixo da barra longitudinal e a face lateral (m)
t1, t2, t3, t4 = b/2, b/2, b/2, b/2  # Espessura máxima dos painéis (m)

### Quantidade das armaduras:
s = 1            # Espaçamento da armadura transversal (m)

### Propriedades Mecânicas dos aços:
fyk = 500    # Tensão de escoamento do aço de projeto (MPa)
fly = 500    # Tensão de escoamento da armadura longitudinal (MPa)
fty = 500    # Tensão de escoamento da armadura transversal (MPa)
Es = 210000  # Módulo de elasticidade dos aços (MPa)
gammas = 1.15

### Propriedades Mecânicas do concreto:
fck = 40        # Resistência característica do concreto comprimido (MPa)
e0 = -2*10**-3  # Deformação de compressão última do concreto (1/1000)
ecr = 0.1       # Deformação de tração que o concreto fissura (1/1000)
ecr0 = 4.5      # Deformação de tração limite do concreto (1/1000)
eds1 = -0.01    # Deformação de compressão inicial do painel 1 (1/1000)
gammac = 1.4

### Relação dos outros esforços com o momento torsor:
MyTx = 0  # Momento Fletor y / Momento Torsor x
MzTx = 0  # Momento Fletor z / Momento Torsor x
#VyTx = 0  # Esforço Cortante y / Momento Torsor x
VzTx = 0  # Esforço Cortante z / Momento Torsor x
NxTx = 0  # Esforço Normal x / Momento Torsor x

### **2. Cálculos preliminares**

In [ ]:
### Propriedades Geométricas:

def PropriedadesGeometricas(b, h, t1, t2, t3, t4):

  # Eq. (4.7) - Área inclusa pelo perímetro externo da seção (m2):
  Acp = b*h

  # Eq.(4.5) - Área bruta da seção (m2):
  Ag = (b-t1)*t4 + (h-t2)*t1 + (b-t3)*t2 + (h-t4)*t3

  # Eq.(4.6) - Perímetro externo da seção (m):
  pcp = 2*(b+h)

  he = min(Acp/pcp,b-2*c1)  # Espessura equivalente (item 17.5.1.4.1)

  # Área inclusa e perímetro da linha média do fluxo (item 17.5.1.4.1):
  if he > 2*c1:
    xe, ye = b - he, h - he  # Caso 1
    Ae, ue = xe*ye, 2*(xe + ye)
  else:
    he = min(he, min(b,h) - 2*c1)
    xe, ye = b - 2*c1, h - 2*c1  # Caso 2
    Ae, ue = xe*ye, 2*(xe + ye)

  return Acp, Ag, pcp, he, xe, ye, Ae, ue


### Propriedades do Concreto:

def PropriedadesConcreto(fck):

  Acp, Ag, pcp, he, xe, ye, Ae, ue = PropriedadesGeometricas(b, h, t1, t2, t3, t4)

  # Eq.(3.23) - Resistência máxima a tração do concreto (MPa):
  fcr = 0.5*Ag/Acp*np.sqrt(fck)

  # ACI 318-14 - Momento torsor de fissuração:
  Tcr = np.sqrt(fck)/3*Acp**2/pcp

  # Resistência de cálculo à compressão do concreto (item 12.3.3):
  fcd = fck/gammac

  # Resistência média à tração do concreto (item 8.2.5):
  if fck < 50:
    fctm = 0.3*fck**(2/3)
  else:
    fctm = 2.12*np.ln(1+0.11*fck)

  # Resistência característica inferior do concreto à tração (item 8.2.5):
  fctkinf = 0.7*fctm

  # Resistência de projeto à tração do concreto (item 17.4.2.2):
  fctd = fctkinf/gammac

  return fcr, Tcr, fcd, fctm, fctkinf, fctd


### Propriedades do Aço:

def PropriedadesAço(fly, fty, Es, fyk):

  # Eq.(2.13) - Deformação de escoamento longitudinal (1/1000):
  ely = fly/Es*1000

  # Eq.(2.13) - Deformação de escoamento transversal:
  ety = fty/Es

  # Tensão de escoamento do aço de projeto (item 8.3.6):
  fyd = fyk/gammas

  return ely, ety, fyd

### **3. Função do Combined Action - Softened Truss Model**

In [ ]:
def CASTM(x, eds1, VyTx, Al1, Al2, Al3, Al4, At):

  #####################
  # Variáveis globais #
  #####################

  global theta

  #####################
  # Cálculos iniciais #
  #####################

  fcr, Tcr, _, _, _, _ = PropriedadesConcreto(fck)

  ely, ety, _ = PropriedadesAço(fly, fty, Es, fyk)

  # Eq.(3.22a) - Módulo de elasticidade do concreto tracionado (MPa):
  Ec = fcr/ecr*1000 # Obs: Variável local usada nessa função.

  ###################################################
  # Combined Action - Softened Truss Model (CA-STM) #
  ###################################################

  # Inicialização da função resíduo:
  Fun = np.zeros(16)

  # Relação dos outros esforços com o momento torsor:
  Tx = Tcr*x[15]
  My, Mz = MyTx*Tx, MzTx*Tx
  Vy, Vz = VyTx*Tx, VzTx*Tx
  Nx = NxTx*Tx

  # Eq.(3.3) e Eq.(3.4) - Espessura do fluxo e deformação interna do painel:
  td1, ea1 = (x[11]*t1/2, 0) if x[11] < 2 else (t1, (x[11] - 2)*eds1/1000)
  td2, ea2 = (x[12]*t2/2, 0) if x[12] < 2 else (t2, (x[12] - 2)*x[0]/1000)
  td3, ea3 = (x[13]*t3/2, 0) if x[13] < 2 else (t3, (x[13] - 2)*x[1]/1000)
  td4, ea4 = (x[14]*t4/2, 0) if x[14] < 2 else (t4, (x[14] - 2)*x[2]/1000)

  # Eq.(4.8) e Eq. (4.9) - Base e altura da área do braço de alavanca:
  b0 = b - (td1 + td3)/2
  h0 = h - (td2 + td4)/2

  # Eq.(4.10) - Área do braço de alavanca:
  A0 = b0*h0

  # Eq.(3.1) - Deformação principal de compressão nos painéis:
  ed1 = (eds1/1000 + ea1)/2
  ed2 = (x[0]/1000 + ea2)/2
  ed3 = (x[1]/1000 + ea3)/2
  ed4 = (x[2]/1000 + ea4)/2

  # Eq.(3.27) e (3.28) – Curvaturas longitudinais:
  fiL13 = (x[7]/1000 -  x[9]/1000)/b0
  fiL24 = (x[8]/1000 - x[10]/1000)/h0

  # Eq.(3.18) - 1º Princípio da invariância das deformações:
  et1 = x[3]/1000 + ed1 -  x[7]/1000
  et2 = x[4]/1000 + ed2 -  x[8]/1000
  et3 = x[5]/1000 + ed3 -  x[9]/1000
  et4 = x[6]/1000 + ed4 - x[10]/1000

  # Eq.(3.25) e (3.26) - Curvaturas transversais:
  fiT13 = (et1 - et3)/b0
  fiT24 = (et2 - et4)/h0

  # Eq.(3.19) - Coeficientes de amolecimento:
  zeta1 = 0.9/np.sqrt(1 + 0.6*x[3])
  zeta2 = 0.9/np.sqrt(1 + 0.6*x[4])
  zeta3 = 0.9/np.sqrt(1 + 0.6*x[5])
  zeta4 = 0.9/np.sqrt(1 + 0.6*x[6])

  # Eq.(3.21) - Relação entre a resistência à compressão no pico e a média:
  k11a = ((eds1/1000)/e0*(1-(eds1/1000)/3/e0)-ea1**2/(eds1/1000)/e0*(1-ea1/3/e0))*(eds1/1000)/((eds1/1000)-ea1)
  k12a = ((x[0]/1000)/e0*(1-(x[0]/1000)/3/e0)-ea2**2/(x[0]/1000)/e0*(1-ea2/3/e0))*(x[0]/1000)/((x[0]/1000)-ea2)
  k13a = ((x[1]/1000)/e0*(1-(x[1]/1000)/3/e0)-ea3**2/(x[1]/1000)/e0*(1-ea3/3/e0))*(x[1]/1000)/((x[1]/1000)-ea3)
  k14a = ((x[2]/1000)/e0*(1-(x[2]/1000)/3/e0)-ea4**2/(x[2]/1000)/e0*(1-ea4/3/e0))*(x[2]/1000)/((x[2]/1000)-ea4)

  k11b = (2*(eds1/1000)*e0 - (eds1/1000)**2)/(e0**2)
  k12b = (2*(x[0]/1000)*e0 - (x[0]/1000)**2)/(e0**2)
  k13b = (2*(x[1]/1000)*e0 - (x[1]/1000)**2)/(e0**2)
  k14b = (2*(x[2]/1000)*e0 - (x[2]/1000)**2)/(e0**2)

  k11 = k11a if ea1 > (eds1/1000) else k11b
  k12 = k12a if ea2 > (x[0]/1000) else k12b
  k13 = k13a if ea3 > (x[1]/1000) else k13b
  k14 = k14a if ea4 > (x[2]/1000) else k14b

  # Eq.(3.20) - Tensão de compressão principal no concreto:
  sigmad1 = -zeta1*fck*k11
  sigmad2 = -zeta2*fck*k12
  sigmad3 = -zeta3*fck*k13
  sigmad4 = -zeta4*fck*k14

  # Eq.(3.22a) e Eq.(3.22d) - Tensão de tração principal no concreto:
  sigmar1 = Ec*(x[3]/1000) if x[3] < ecr else fcr*np.exp(-350*(x[3]-ecr)/1000)
  sigmar2 = Ec*(x[4]/1000) if x[4] < ecr else fcr*np.exp(-350*(x[4]-ecr)/1000)
  sigmar3 = Ec*(x[5]/1000) if x[5] < ecr else fcr*np.exp(-350*(x[5]-ecr)/1000)
  sigmar4 = Ec*(x[6]/1000) if x[6] < ecr else fcr*np.exp(-350*(x[6]-ecr)/1000)

  # Eq.(3.24) - Tensão no aço transversal:
  A, F = 0.002, 4
  Bt = (1 - A)/ety
  ft1 = Es*et1*(A + (1-A)/(1 + (Bt*et1)**F)**(1/F))
  ft2 = Es*et2*(A + (1-A)/(1 + (Bt*et2)**F)**(1/F))
  ft3 = Es*et3*(A + (1-A)/(1 + (Bt*et3)**F)**(1/F))
  ft4 = Es*et4*(A + (1-A)/(1 + (Bt*et4)**F)**(1/F))

  # Eq.(4.1) - Seno de alfaD ao quadrado:
  sin21 = (( x[7]/1000) - ed1)/((x[3]/1000) - ed1)
  sin22 = (( x[8]/1000) - ed2)/((x[4]/1000) - ed2)
  sin23 = (( x[9]/1000) - ed3)/((x[5]/1000) - ed3)
  sin24 = ((x[10]/1000) - ed4)/((x[6]/1000) - ed4)

  # Eq.(4.2) - Cosseno de alfaD ao quadrado:
  cos21 = ((x[3]/1000) -  (x[7]/1000))/((x[3]/1000) - ed1)
  cos22 = ((x[4]/1000) -  (x[8]/1000))/((x[4]/1000) - ed2)
  cos23 = ((x[5]/1000) -  (x[9]/1000))/((x[5]/1000) - ed3)
  cos24 = ((x[6]/1000) - (x[10]/1000))/((x[6]/1000) - ed4)

  # Eq.(4.3) - Produto do seno e cosseno de alfaD:
  sincos1 = np.sqrt((( x[7]/1000) - ed1)*(abs(et1 - ed1)))/((x[3]/1000) - ed1)
  sincos2 = np.sqrt((( x[8]/1000) - ed2)*(abs(et2 - ed2)))/((x[4]/1000) - ed2)
  sincos3 = np.sqrt((( x[9]/1000) - ed3)*(abs(et3 - ed3)))/((x[5]/1000) - ed3)
  sincos4 = np.sqrt(((x[10]/1000) - ed4)*(abs(et4 - ed4)))/((x[6]/1000) - ed4)

  # Eq.(3.15) - Equação de equilíbrio transversal:
  Fun[0] = sigmad1*sin21 + sigmar1*cos21 + ft1*(At/td1/s)
  Fun[1] = sigmad2*sin22 + sigmar2*cos22 + ft2*(At/td2/s)
  Fun[2] = sigmad3*sin23 + sigmar3*cos23 + ft3*(At/td3/s)
  Fun[3] = sigmad4*sin24 + sigmar4*cos24 + ft4*(At/td4/s)

  # Eq.(3.9) - Cálculo dos Fluxos:
  q1 = Tx/2/A0 + Vy/2/h0
  q2 = Tx/2/A0 + Vz/2/b0
  q3 = Tx/2/A0 - Vy/2/h0
  q4 = Tx/2/A0 - Vz/2/b0

  # Eq.(3.17) - Deformações de corte nos painéis:
  gammalt1 = 2*((x[3]/1000) - ed1)*sincos1*np.sign(q1)
  gammalt2 = 2*((x[4]/1000) - ed2)*sincos2*np.sign(q2)
  gammalt3 = 2*((x[5]/1000) - ed3)*sincos3*np.sign(q3)
  gammalt4 = 2*((x[6]/1000) - ed4)*sincos4*np.sign(q4)

  # Eq.(3.33) - Ângulo TETA:
  theta = ((gammalt1 + gammalt3)*h0 + (gammalt2 + gammalt4)*b0)/2/A0

  # Eq.(3.2) e Eq.(3.32) - Curvatura da biela:
  curv1a = -((eds1/1000) - ea1)/td1
  curv2a = -((x[0]/1000) - ea2)/td2
  curv3a = -((x[1]/1000) - ea3)/td3
  curv4a = -((x[2]/1000) - ea4)/td4

  curv1b = theta*2*sincos1 - fiL13*cos21 - fiT13*sin21
  curv2b = theta*2*sincos2 - fiL24*cos22 - fiT24*sin22
  curv3b = theta*2*sincos3 + fiL13*cos23 + fiT13*sin23
  curv4b = theta*2*sincos4 + fiL24*cos24 + fiT24*sin24

  Fun[4] = curv1a - curv1b
  Fun[5] = curv2a - curv2b
  Fun[6] = curv3a - curv3b
  Fun[7] = curv4a - curv4b

  # Eq.(3.10) e Eq.(3.16) - Tensão de cisalhamento:
  tal1a = q1/td1
  tal2a = q2/td2
  tal3a = q3/td3
  tal4a = q4/td4

  tal1b = (-sigmad1 + sigmar1)*sincos1*np.sign(q1)
  tal2b = (-sigmad2 + sigmar2)*sincos2*np.sign(q2)
  tal3b = (-sigmad3 + sigmar3)*sincos3*np.sign(q3)
  tal4b = (-sigmad4 + sigmar4)*sincos4*np.sign(q4)

  Fun[8]  = tal1a - tal1b
  Fun[9]  = tal2a - tal2b
  Fun[10] = tal3a - tal3b
  Fun[11] = tal4a - tal4b

  # Eq.(3.24) - Tensão no aço longitudinal:
  Bl = (1 - A)/ely
  fl1 =  Es*x[7]/1000*(A + (1-A)/(1 +  (Bl*x[7])**F)**(1/F))
  fl2 =  Es*x[8]/1000*(A + (1-A)/(1 +  (Bl*x[8])**F)**(1/F))
  fl3 =  Es*x[9]/1000*(A + (1-A)/(1 +  (Bl*x[9])**F)**(1/F))
  fl4 = Es*x[10]/1000*(A + (1-A)/(1 + (Bl*x[10])**F)**(1/F))

  # Eq.(3.14) - Equação de equilíbrio longitudinal:
  sigmal1 = sigmad1*cos21 + sigmar1*sin21 + fl1*Al1/td1/h0
  sigmal2 = sigmad2*cos22 + sigmar2*sin22 + fl2*Al2/td2/b0
  sigmal3 = sigmad3*cos23 + sigmar3*sin23 + fl3*Al3/td3/h0
  sigmal4 = sigmad4*cos24 + sigmar4*sin24 + fl4*Al4/td4/b0

  # Eq.(3.11) - Momento fletor no eixo Y:
  Fun[12] = (sigmal3*td3*h0 - sigmal1*td1*h0)*b0/2 - My

  # Eq.(3.12) - Momento fletor no eixo Z:
  Fun[13] = (sigmal4*td4*b0 - sigmal2*td2*b0)*h0/2 - Mz

  # Eq.(3.13) - Esforço Normal em X:
  Fun[14] = sigmal1*td1*h0 + sigmal2*td2*b0 + sigmal3*td3*h0 + sigmal4*td4*b0 - Nx

  # Eq.(3.30) - Compatibilização das deformações longitudinais:
  ecl13 = (x[7]/1000 +  x[9]/1000)/2
  ecl24 = (x[8]/1000 + x[10]/1000)/2
  Fun[15] = ecl13 - ecl24

  return Fun

### **4. Procedimento de solução CA-STM**

In [ ]:
#################################
# Função do procedimento CA-STM #
#################################

def ProcedimentoCASTM(VyTx, Al1, Al2, Al3, Al4, At):

  #########################
  # Cálculos preliminares #
  #########################

  Acp, Ag, pcp, _, _, _, _, _ = PropriedadesGeometricas(b, h, t1, t2, t3, t4)

  fcr, Tcr, _, _, _, _ = PropriedadesConcreto(fck)

  # NBR-6118/2007 - Módulo de elasticidade secante do concreto (MPa):
  Ecc = 0.85*5600*np.sqrt(fck) # Obs: Variável local só para o chute inicial.

  #################################
  # Cálculo da estimativa inicial #
  #################################

  # Inicialização da estimativa:
  x0 = np.zeros(16)

  # Preenchimento do vetor da estimativa:
  for i in range(3):
    x0[i] = eds1
  for i in range(3, 7):
    x0[i] = -Ecc*ecr/2/fcr*eds1*10**-3 # Diferente do ACI.
    x0[i+4] = 0
    x0[i+8] = 1
  x0[15] = -Ecc/2*Acp**2/pcp*eds1*10**-3 # Diferente do ACI.

  #########################################
  # Parâmetros do procedimento de solução #
  #########################################

  # Limite inferior da solução:
  lb = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
  lb[0], lb[1], lb[2] = -100, -100, -100

  # Limite superior da solução:
  ub = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 0])
  ub[3], ub[ 4], ub[ 5] = 100, 100, 100
  ub[6], ub[ 7], ub[ 8] = 100, 100, 100
  ub[9], ub[10], ub[15] = 100, 100, 100

  # Vetor com os limites inferior e superior:
  bounds = (lb, ub)

  # Número pontos avaliados:
  nmax = 400

  # Tamanho do passo da análise:
  passo = 0.01

  ###########################
  # Procedimento de solução #
  ###########################

  # Inicialização do vetor com a curva Txtheta:
  thetacurva = [0]
  Tcurva = [0]

  # Inicialização do iterador:
  i = 1

  # Iteração para cada ponto avaliado:
  while i < nmax:

    # Solução do problema não-linear:
    sol = least_squares(CASTM, x0, bounds=bounds, args=(eds1-i*passo, VyTx, Al1, Al2, Al3, Al4, At), method='trf', max_nfev= 300)

    # Atualização da estimativa inicial:
    x0 = x = sol.x

    # Verificação do esmagamento do concreto:
    DeformacaoMaxCompressao = np.array([eds1-i*passo, x[0], x[1], x[2]])
    if np.min(DeformacaoMaxCompressao) < -3.5:
      #print('Critério de parada: O CONCRETO ESMAGOU! \n')
      break

    # Verificação do torsor último:
    if (np.min(DeformacaoMaxCompressao) < -3.5/2 and Tcurva[-1] > x[15]*Tcr*1000):
      #print('Critério de parada: ATINGIU O TORSOR ÚLTIMO! \n')
      break

    # Salvar solução nos vetor T e theta:
    thetacurva.append(theta)
    Tcurva.append(x[15]*Tcr*1000)

    # Incremento do iterador:
    i = i + 1

  ##################################################
  # Cálculo do divisor utilizado no banco de dados #
  ##################################################

  # Rigidez à torção elástica teórica (kN.m2):
  Gc = Ecc*1000/2/1.2
  xb, xa = min(b,h), max(b,h)
  RigTeoE = Gc*(1/3 - 0.21*xb/xa*(1 - xb**4/12/xa**4))*xa*xb**3

  # Rigidez à torção última obtida no modelo (KN.m2):
  RigNumU = Tcurva[-1]/thetacurva[-1]

  # Divisor da rigidez à torção usado no banco de dados (RigTeoE/RigNumU):
  DivData = RigTeoE/RigNumU

  '''Silva (2016):    Rig. Elástica do modelo / Rig. última do modelo.
     Medeiros (2026): Rig. Elástica Teórica   / Rig. última do modelo (DivData)'''

  return DivData, Tcurva, thetacurva, RigTeoE, RigNumU

### **5. Função para comparação com os resultados do Mestrado**

In [ ]:
######################################################
# Função de comparação com os resultados do Mestrado #
######################################################

def FuncaoComparacaoMestrado():

  # Chamada do procedimento de solução CA-STM:
  DivData, Tcurva, thetacurva, RigTeoE, RigNumU = ProcedimentoCASTM(VyTx, Al1, Al2, Al3, Al4, At)

  ##############################################
  # Cálculo das rigidezes e divirores à torção #
  ##############################################

  # Rigidez à torção última obtida no experimento:
  RigExpU = max(Texp)/thetaexp[np.argmax(Texp)]

  # Rigidez à torção elástica obtida no modelo:
  RigNumE = Tcurva[1]/thetacurva[1]

  # Rigidez à torção elástica obtida no experimento:
  RigExpE = Texp[1]/thetaexp[1]

  # Divisor da rigidez à torção do modelo:
  DivNum = RigNumE/RigNumU # Obs: Conforme Silva (2016).

  # Divisor da rigidez à torção do experimento:
  DivExp = RigExpE/RigExpU

  #####################################
  # Cálculo das tabelas de resultados #
  #####################################

  # Dados das tabelas:
  dados1 = [['Rigidez Elástica teórica',        round(RigTeoE,0), RigTeoEPub],
            ['Rigidez Elástica do Experimento', round(RigExpE,0), RigExpEPub],
            ['Rigidez Elástica do Modelo',      round(RigNumE,0), RigNumEPub],
            ['Rigidez Última do Experimento',   round(RigExpU,0), RigExpUPub],
            ['Rigidez Última do Modelo',        round(RigNumU,0), RigNumUPub]]

  dados2 = [['Experimento', round(DivExp,1), DivExpPub],
            ['Modelo',      round(DivNum,1), DivNumPub]]

  # Cabeçalhos das tabelas:
  cabecalho1 = ['Rigidez à torção (kN.m2)', 'Google Colab', 'Silva (2016)']
  cabecalho2 = ['Divisor à torção',         'Google Colab', 'Silva (2016)']

  # Criar as tabelas:
  tabela1 = tabulate(dados1, headers=cabecalho1)
  tabela2 = tabulate(dados2, headers=cabecalho2)

  ############
  # Plotagem #
  ############

  # Plotagem da curva gráfico:
  plt.plot(thetaexp, Texp, marker='s', linestyle='-', color='red', label = LabelExp)
  plt.plot(thetapub, Tpub, linestyle='--', color='k', label = LabelPub)
  plt.plot(thetaNewM, TNewM, linestyle='-', color='y', label = LabelNewM)
  plt.plot(thetacurva, Tcurva, linestyle='--', color='blue', label = LabelNum)

  # Adicionar legendas:
  plt.xlabel('Rotação axial (rad/m)')
  plt.ylabel('Momento Torsor (kN.m)')
  plt.title('Curva Torsor-Rotação')

  # Adicionar grid
  plt.grid(True)

  # Delimitar intervalos dos eixos x e y
  plt.xlim(0, )
  plt.ylim(0, )

  # Adicionar legenda
  plt.legend(loc = 'lower right')

  # Mostrar o gráfico
  plt.show()

  # Imprimir a tabela:
  print(tabela1, "\n")
  print(tabela2)

In [ ]:
FuncaoComparacaoMestrado()

# **Dimensionamento de seções retangulares para Torção e Cortante**

Algoritmo para o dimensionamento de seções retangulares de concreto armado, submetidas a torção e esforço cortante, segundo a NBR-6118:2023 e ARAÚJO (2014).

**REFERÊNCIAS**

*   ARAÚJO, J. M., 2014. “Curso de Concreto Armado”, Rio Grande: Dunas, v.4, 4.ed. 2014.
*   ASSOCIAÇÃO BRASILEIRA DE NORMAS TÉCNICAS. “Projeto de Estruturas de Concreto, NBR – 6118 (2023)”. ABNT, Rio de Janeiro, Brasil, 2023.

### **1. Função de Dimensionamento**

In [ ]:
def DimensionamentoTorcaoCortante(b, h, fck, βt, βt_dim, βv):

  ##############################
  # Propriedades dos materiais #
  ##############################

  _, _, fcd, fctm, _, fctd = PropriedadesConcreto(fck)
  _, _, fyd = PropriedadesAço(fly, fty, Es, fyk)
  _, _, _, he, xe, ye, Ae, ue = PropriedadesGeometricas(b, h, t1, t2, t3, t4)

  ############################
  # Dimensionamento à torção #
  ############################

  ### Momento torsor de esmagamento das diagonais comprimidas (item 17.5.1.5):
  alfav2 = 1 - fck/250  # Com fck expresso em MPa
  Trd2 = 0.5*alfav2*Ae*he*fcd

  ### Momento torsor de projeto:
  Tsd = βt * Trd2  # kN.m
  Tsd_dim = βt_dim * Trd2  # kN.m

  ### Armadura longitudinal de torção (item 17.5.1.6):
  Asl = Tsd_dim/(2*Ae*fyd)*ue
  Aslmin = 0.2*fctm/fyk*he*ue
  Asl = max(Asl, Aslmin)

  ### Armadura longitudinal por face:
  Al1 = Asl*ye/ue  # Área da armadura longitudinal no painel 1 (m2)
  Al2 = Asl*xe/ue  # Área da armadura longitudinal no painel 2 (m2)
  Al3 = Asl*ye/ue  # Área da armadura longitudinal no painel 3 (m2)
  Al4 = Asl*xe/ue  # Área da armadura longitudinal no painel 4 (m2)

  ### Armadura transversal de torção (item 17.5.1.6):
  A90s = Tsd_dim/(2*Ae*fyd)

  ###############################
  # Dimensionamento ao cortante #
  ###############################

  ### Esforço cortante de esmagamento das diagonais comprimidas, Modelo 1 (item 17.4.2.2):
  d = h - c1
  Vrd2 = 0.27*alfav2*fcd*b*d

  ### Esforço cortante de projeto:
  Vsd = βv * Vrd2

  ### Verificação do esmagamento da diagonal comprimida de concreto (item 17.5.1.5):
  if (Vsd/Vrd2 + Tsd/Trd2 > 1):
    print('Esmagou!')

  ### Contribuição do concreto na resistência ao cortante, Modelo 1 (item 17.4.2.2):
  Vc0 = 0.6*fctd*b*d

  ### Dimensionamento da armadura de cortante (item 17.4.2.2):
  Asws = (Vsd - Vc0)/(0.9*d*fyd)
  Asws = max(Asws, 0)

  ### Armadura transversal por face:
  At = Asws/2 + A90s
  Aswsmin = 0.2*fctm/fyk*b  # item 17.4.1.1.1
  At = max(At, Aswsmin/2)

  return Al1, Al2, Al3, Al4, At, Trd2, Vrd2, Tsd, Vsd

### **2. Função de Cálculo do βtcr**

In [ ]:
def Calculo_βtcr(fck):

  ##############################
  # Propriedades dos materiais #
  ##############################

  _, _, _, he, _, _, Ae, _ = PropriedadesGeometricas(b, h, t1, t2, t3, t4)
  _, Tcr, fcd, _, _, _ = PropriedadesConcreto(fck)

  ##############################
  #      Cálculo do βtcr       #
  ##############################

  ### Momento torsor de esmagamento das diagonais comprimidas (item 17.5.1.5):
  alfav2 = 1 - fck/250
  Trd2 = 0.5*alfav2*Ae*he*fcd

  ### Beta do torsor de fissuração:
  βtcr = Tcr/Trd2

  return βtcr

### **3. Análise dos dados de rigidez**

In [ ]:
step = 0.1  # Passo das iterações

def analise_rigidez():

  ########################
  # Inicialização geral  #
  ########################

  # Definindo o intervalo de βt:
  βtcr = Calculo_βtcr(fck)                      # Calcula o valor de βtcr
  βt_range   = np.arange(step, 1 + step, step)     # βt variando de 0 até 1
  βt_range = βt_range[βt_range <= 1]            # Garante que βt não ultrapasse 1

  resultados = []                                  # Inicialização de lista para armazenar os resultados obtidos em cada iteração
  Tcurva_thetacurva = {βt: [] for βt in βt_range}  # Inicialização de dicionário com uma chave para cada valor de βt

  ###############################
  # Loop sobre os valores de βt #
  ###############################

  for βt in βt_range:

    # Definindo o intervalo de βv:
    βvmax = 1 - βt  # βv máximo associado ao βt atual

    # Intervalo de βv variando de 0 até βvmax:
    if step > βvmax: βv_range = np.array([0, βvmax]) # Condição para desconsiderar o "step" quando ele for maior que βvmax
    else:            βv_range = np.arange(0, βvmax + step, step)

    # Garantindo a inclusão de βvmax quando for necessário:
    if not np.isclose(βv_range, βvmax, atol=1e-8).any():
      βv_range = np.append(βv_range, βvmax)

    # Eliminando valores acima de βvmax e duplicatas:
    βv_range = βv_range[βv_range <= βvmax + 1e-8]
    βv_range = np.unique(βv_range)

    ###############################
    # Loop sobre os valores de βv #
    ###############################

    for βv in βv_range:

      ### DIMENSIONAMENTO:
      if βt < βtcr: βt_dim = βtcr # Ajustando βt apenas para o dimensionamento das armaduras
      else: βt_dim = βt
      Al1, Al2, Al3, Al4, At, Trd2, Vrd2, Tsd, Vsd = DimensionamentoTorcaoCortante(b, h, fck, βt, βt_dim, βv)

      Asl  = round((Al1 + Al2 + Al3 + Al4)*1e4, 2)  # Área da armadura longitudinal total (cm²)
      Ast  = round(At*1e4, 2)                       # Área da armadura transversal total (cm²)
      VyTx = Vsd / Tsd                              # Razão entre o esforço cortante e momento torsor

      '''Obs: Apesar do modelo do Greene (2006) considerar 6 esforços internos na seção (caso geral),
              nesse trabalho, estamos considerando que a seção está submetida a torção + um cortante Vy.
              Em um próximo trabalho, vamos considerar T+Vy+Mx, e em um seguinte, T+Vy+Mx+N.'''

      ### CÁLCULOS DE RIGIDEZ UTILIZANDO A FUNÇÃO PROCEDIMENTO DE CA-STM:
      DivData, Tcurva, thetacurva, RigTeoE, RigNumU = ProcedimentoCASTM(VyTx, Al1, Al2, Al3, Al4, At)

      ### ARMAZENANDO AS CURVAS DE RIGIDEZ PARA ESTE βt:
      Tcurva_thetacurva[βt].append((Tcurva, thetacurva))

      ### ARMAZENANDO OS RESULTADOS DESTA ITERAÇÃO:
      vals = [round(βt, 2), round(βv, 2), round(Tsd * 1000, 1), round(Vsd * 1000, 1), Asl, Ast,
        round(RigTeoE, 1), round(RigNumU, 1), round(DivData, 1), round(100 / DivData, 1)]
      resultados.append(vals)

  return resultados, Tcurva_thetacurva, Trd2, βtcr

In [ ]:
resultados, Tcurva_thetacurva, Trd2, βtcr = analise_rigidez()

### **4. Plotagem das curvas Torsor-Rotação agrupadas por βt**

In [ ]:
def plotagem_curvas_torsor():

  #############################################################
  # Plotagem das curvas Torsor-Rotação agrupadas segundo o βt #
  #############################################################

  # Iterando sobre cada βt e suas curvas correspondentes do dicionário Tcurva_thetacurva
  for βt, curvas in Tcurva_thetacurva.items():
    if not curvas: continue # Verifica se o grupo de curvas para este βt está vazio, caso esteja, pula para o próximo βt

    ###############################
    #    Geração dos gráficos     #
    ###############################

    # Criando uma figura para cada βt:
    fig, ax = plt.subplots(figsize=(7, 7))

    # Definindo uma paleta de cores:
    paleta_cores = ["#B36B00", "#3686B3", "#007B5C", "#B3AA00", "#004F80", "#9E3E00", "#994F73", "#666666"]

    # Filtrando os valores de βv para cada grupo de βt:
    lista_βv = [resultado[1] for resultado in resultados]

    for i, (Tcurva, thetacurva) in enumerate(curvas):  # Itera sobre as curvas para cada grupo de βt
        βv_atual = lista_βv[i]                         # Filtrando o βv atual dentro da lista de valores
        color = paleta_cores[i % len(paleta_cores)]    # Escolhe a cor da curva atual dentro da paleta de cores
        ax.plot(thetacurva, Tcurva, color = color, label=f'βv = {βv_atual:.2f}')  # Plota a curva com cor definida

    # Personalizando o gráfico:
    ax.set_title(f'Curvas Torsor-Rotação para βt = {βt:.2f}', fontsize=14, pad=10)  # Título do gráfico com o valor de βt
    ax.set_xlabel('Rotação axial (rad/m)', fontsize=12)  # Legenda do eixo x
    ax.set_ylabel('Momento Torsor (kN.m)', fontsize=12)  # Legenda do eixo y
    ax.set_xlim(0, None)  # Limita o intervalo do eixo x
    ax.set_ylim(0, None)  # Limita o intervalo do eixo y
    ax.grid(True, alpha=0.6)  # Adiciona a malha (grid) no gráfico
    ax.legend(fontsize=10, title_fontsize=11, loc='lower right', framealpha=1)  # Define e configura a legenda
    plt.show()  # Exibe o gráfico

In [ ]:
plotagem_curvas_torsor()

### **5. Geração da tabela com os dados obtidos**

In [ ]:
##########################################################################
#      Criando uma tabela para exibição e formatação dos resultados      #
##########################################################################

# Criando o DataFrame com os resultados:
df = pd.DataFrame(resultados, columns=[
    'βt', 'βv', 'Tsd \n(kN.m)', 'Vsd \n(kN)', 'Asl \n(cm²)', 'Ast \n(cm²)',
    'GC_elas', 'GC_ult', 'Divisor', 'Rigidez efetiva/\nRigidez elástica'])

# Formatando o DataFrame com as casas decimais apropriadas para cada coluna:
styled_df = df.style.format({
    'βt': '{:.2f}', 'βv': '{:.2f}', 'Tsd \n(kN.m)': '{:.1f}', 'Vsd \n(kN)': '{:.1f}',
    'Asl \n(cm²)': '{:.2f}', 'Ast \n(cm²)': '{:.2f}', 'GC_elas': '{:.1f}', 'GC_ult': '{:.1f}',
    'Divisor': '{:.1f}', 'Rigidez efetiva/\nRigidez elástica': '{:.1f}%'})

# Exibindo o cabeçalho da tabela:
print("\n" + "="*110)
print("RESULTADOS DA ANÁLISE DE RIGIDEZ À TORÇÃO".center(110))
print("="*110 + "\n")

# Gerando uma tabela formatada a partir do dataframe:
table = tabulate(
    df,                     # DataFrame criado antes contendo os dados da análise
    headers='keys',         # Utiliza os nomes das colunas do DataFrame como cabeçalhos da tabela
    tablefmt='fancy_grid',  # Estilo visual da tabela (Utiliza bordas duplas para melhor apresentação)
    showindex=False,        # Não exibe a coluna de índices do DataFrame
    numalign="center",      # Alinhamento centralizado para colunas com valores numéricos
    stralign="center",      # Alinhamento centralizado para colunas com strings
    floatfmt=".2f")         # Formato de exibição dos números com 2 casas decimais

# Exibindo a tabela:
print(table)

# Exibindo o rodapé da tabela:
print("\n" + "="*110)

### **6. Salvando a tabela com os dados obtidos**

In [ ]:
pip install xlsxwriter # Instalação de pacote para manipular arquivos de .xlsx

In [ ]:
# Nome do arquivo Excel
nome_arquivo = 'Resultado_CASTM.xlsx'

# Criando o ExcelWriter com engine xlsxwriter
with pd.ExcelWriter(nome_arquivo, engine='xlsxwriter') as writer:
    # Convertendo o DataFrame sem formatá-lo
    df.to_excel(writer, sheet_name='Resultados', index=False)

    # Acessando workbook e worksheet
    workbook  = writer.book
    worksheet = writer.sheets['Resultados']

    # Formatações personalizadas
    formatacao = workbook.add_format({'num_format': '0.00', 'align': 'center'})

    # Mapeando colunas e aplicando formatação
    colunas_formatadas = {
        'βt':           formatacao,
        'βv':           formatacao,
        'Tsd \n(kN.m)': formatacao,
        'Vsd \n(kN)':   formatacao,
        'Asl \n(cm²)':  formatacao,
        'Ast \n(cm²)':  formatacao,
        'GC_elas':      formatacao,
        'GC_ult':       formatacao,
        'Divisor':      formatacao,
        'Rigidez efetiva/Rigidez elástica': formatacao}

    # Aplicando a formatação às colunas corretas
    for i, col_name in enumerate(df.columns):
        if col_name in colunas_formatadas:
            worksheet.set_column(i, i, 20, colunas_formatadas[col_name])
        else:
            worksheet.set_column(i, i, 20)

### **7. Curva de nível de Rigidez à torção**

In [ ]:
def curvas_nivel():

  ###################################
  #    Propriedades dos materiais   #
  ###################################

  _, Tcr, _, _, _, _ = PropriedadesConcreto(fck)

  ###################################
  #  Configuração da curva de nível #
  ###################################

  # Extraindo colunas relevantes do DataFrame para a geração da curva de nível:
  x = df['βv']
  y = df['βt']
  z = df['Divisor']

  # Criando a malha de valores (grid) para interpolação:
  xi = np.linspace(x.min(), x.max(), 100)
  yi = np.linspace(y.min(), y.max(), 100)
  xi, yi = np.meshgrid(xi, yi)

  # Interpolando os valores de z sobre a malha utilizando o método cúbico:
  zi = griddata((x, y), z, (xi, yi), method='cubic')

  ###################################
  #    Plotagem da curva de nível   #
  ###################################

  # Inicializando a figura:
  fig, ax = plt.subplots(figsize=(7, 7))

  # Gerando o mapa de contorno (curvas de nível):
  contour = ax.contourf(xi, yi, zi, levels=15, cmap='viridis')
  plt.colorbar(contour, label='DIVISOR')  # Barra de cores com legenda

  # Plotando a Curva de interação Torsor-Cortante:
  ax.plot([0, 1], [1, 0], linestyle='--', color='black', label='Curva de interação Torsor-Cortante')

  # Plotando a reta com o Beta do torsor de fissuração:
  ax.axhline(y = βtcr, color='red', linestyle='-.', label=rf'$\beta_{{Tcr}} = {βtcr:.2f}$')

  # Definindo os dados da legenda:
  legenda_Tcr_Trd2 = rf"$T_{{cr}} = {Tcr * 1000:.2f}$ kNm, $T_{{rd2}} = {Trd2 * 1000:.2f}$ kNm"  # Extrai os valores do Tcr e Trd2
  #legenda_divisor = rf"Divisor de Silva (2016) = {DivNumPub}" # Extrai o valor do Dividor de Silva (2016)

  # Adicionando o valor de βtcr próximo à reta plotada:
  ax.text(0.82, βtcr + 0.02, rf'$\beta_{{Tcr}} = {βtcr:.2f}$', fontsize=10, color='red')

  # Personalizando o gráfico:
  ax.set_title('Curva de Nível: Tsd/Trd2 x Vsd/Vrd2', fontsize=14, pad=10) # Título do gráfico
  ax.set_xlabel('Vsd/Vrd2', fontsize=12) # Legenda do eixo x
  ax.set_ylabel('Tsd/Trd2', fontsize=12) # Legenda do eixo y
  ax.set_xlim(0, 1) # Limita o intervalo do eixo x
  ax.set_ylim(0, 1) # Limita o intervalo do eixo y
  ax.set_aspect('equal', adjustable='box') # Mantém escala igual sem ignorar limites
  ax.grid(True, alpha = 0.6) # Adiciona a malha (grid) no gráfico
  #ax.legend(title = legenda_Tcr_Trd2 + '\n   ' + legenda_divisor, fontsize=10, loc='upper right', framealpha=1)
  ax.legend(title = legenda_Tcr_Trd2, fontsize=10, loc='upper right', framealpha=1)

  plt.show() # Exibe o gráfico

In [ ]:
curvas_nivel()